In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [18]:
def read_email_tool(email_id:str)->str:
    """mock fucntion to read an email by its ID"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to send an email"""
    return f"Email send to {recipient} with subject {subject}"

In [19]:
checkpointer = InMemorySaver()

In [20]:
middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email_tool":{
            "allowed_decisions":["approve","edit","reject"]
        },
        "read_email_tool":False,
        
    }
)

In [21]:
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[read_email_tool, send_email_tool],
    middleware=[middleware],
    checkpointer=checkpointer
)

In [23]:
from langchain_core.messages import HumanMessage
config={"configurable":{"thread_id":"test-approve"}}
#step 1 :Request
result=agent.invoke(
    {"messages":[HumanMessage(content="send email to john@example.com with subject 'hello' and body 'how are you'")]},
    config=config
)


In [24]:
result

{'messages': [HumanMessage(content="send email to john@example.com with subject 'hello' and body 'how are you'", additional_kwargs={}, response_metadata={}, id='8cb64f9a-d2ef-41eb-a8be-b4ea8a3e55f5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'v6nhngwas', 'function': {'arguments': '{"body":"how are you","recipient":"john@example.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 315, 'total_tokens': 347, 'completion_time': 0.062974137, 'completion_tokens_details': None, 'prompt_time': 0.018637439, 'prompt_tokens_details': None, 'queue_time': 0.301289591, 'total_time': 0.081611576}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f01f9-5501-7323-ae1f-e834cffced3f-0', tool_calls=[{'name': 'send_email_tool', 'args':

In [26]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: Email sent to john@example.com with subject 'hello' and body 'how are you'


In [27]:
result

{'messages': [HumanMessage(content="send email to john@example.com with subject 'hello' and body 'how are you'", additional_kwargs={}, response_metadata={}, id='8cb64f9a-d2ef-41eb-a8be-b4ea8a3e55f5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'v6nhngwas', 'function': {'arguments': '{"body":"how are you","recipient":"john@example.com","subject":"hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 315, 'total_tokens': 347, 'completion_time': 0.062974137, 'completion_tokens_details': None, 'prompt_time': 0.018637439, 'prompt_tokens_details': None, 'queue_time': 0.301289591, 'total_time': 0.081611576}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f01f9-5501-7323-ae1f-e834cffced3f-0', tool_calls=[{'name': 'send_email_tool', 'args':